In [8]:
import sys
!{sys.executable} -m pip install pandas tiktoken requests python-dotenv -q


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


# Scrape Cost Estimator

Estimate GitHub issue scraping and OpenAI embedding costs before committing to large-scale ingestion.

This notebook:
1. Fetches closed issue metadata from GitHub repos
2. Uses tiktoken to accurately count tokens
3. Calculates embedding costs based on OpenAI pricing

In [9]:
import requests
import json
import os
import time
from dotenv import load_dotenv
import pandas as pd
import tiktoken

load_dotenv()

GITHUB_TOKEN = os.getenv('GITHUB_TOKEN')
BASE_URL = 'https://api.github.com'
HEADERS = {
    'Authorization': f'token {GITHUB_TOKEN}',
    'Accept': 'application/vnd.github+json',
}

# Load tiktoken encoder (cl100k_base is used by text-embedding-3 models)
encoding = tiktoken.get_encoding('cl100k_base')

print('GitHub API and tiktoken ready')

GitHub API and tiktoken ready


## Configuration

In [10]:
# Repos to estimate (owner/repo format)
REPOS = [
    ('swiftlang', 'swift'),
]

# OpenAI embedding model pricing (per 1M tokens)
MODEL_PRICING = {
    'text-embedding-3-small': 0.02,   # $0.02 per 1M tokens (Batch API)
}

print(f'Estimating costs for {len(REPOS)} repo(s)')
print(f'Model: text-embedding-3-small at ${MODEL_PRICING["text-embedding-3-small"]} per 1M tokens')

Estimating costs for 1 repo(s)
Model: text-embedding-3-small at $0.02 per 1M tokens


## Fetch issue counts and estimate text sizes

In [11]:
def count_tokens(text):
    """Count tokens using tiktoken."""
    return len(encoding.encode(text))

def get_closed_issues_sample(owner, repo, sample_size=100):
    """Fetch a sample of closed issues to estimate text size."""
    issues = []
    url = f'{BASE_URL}/repos/{owner}/{repo}/issues'
    page = 1
    
    while len(issues) < sample_size:
        params = {
            'state': 'closed',
            'per_page': 100,
            'page': page,
            'sort': 'created',
            'direction': 'asc',
        }
        
        response = requests.get(url, headers=HEADERS, params=params)
        response.raise_for_status()
        data = response.json()
        
        if not data:
            break
        
        for item in data:
            if 'pull_request' not in item:  # Skip PRs
                issues.append(item)
                if len(issues) >= sample_size:
                    break
        
        page += 1
        time.sleep(0.5)  # Be nice to GitHub API
    
    return issues

def get_total_issue_count(owner, repo):
    """Get total count of closed issues."""
    search_url = f'{BASE_URL}/search/issues'
    search_params = {
        'q': f'repo:{owner}/{repo} is:closed is:issue',
        'per_page': 1,
        'sort': 'created',
        'order': 'desc',
    }
    search_response = requests.get(search_url, headers=HEADERS, params=search_params)
    search_response.raise_for_status()
    return search_response.json()['total_count']

print('Functions ready')

Functions ready


In [12]:
results = []

for owner, repo in REPOS:
    print(f'\nEstimating {owner}/{repo}...')
    
    # Get total count
    total_count = get_total_issue_count(owner, repo)
    print(f'  Total closed issues: {total_count:,}')
    
    # Get sample
    sample = get_closed_issues_sample(owner, repo, sample_size=1000)
    print(f'  Sample size: {len(sample)}')
    
    # Calculate average token count per issue using tiktoken
    total_tokens_sample = 0
    for issue in sample:
        title = issue.get('title', '')
        body = issue.get('body', '') or ''
        # Rough embed_text construction (title + body)
        text = f"TITLE: {title}\nPROBLEM: {body}"
        tokens = count_tokens(text)
        total_tokens_sample += tokens
    
    avg_tokens_per_issue = total_tokens_sample / len(sample)
    
    # Estimate totals
    total_tokens_all = int(avg_tokens_per_issue * total_count)
    
    results.append({
        'repo': f'{owner}/{repo}',
        'total_issues': total_count,
        'sample_size': len(sample),
        'avg_tokens_per_issue': int(avg_tokens_per_issue),
        'total_tokens_sample': total_tokens_sample,
        'total_tokens_estimated': total_tokens_all,
    })
    
    print(f'  Avg tokens/issue (sample): {avg_tokens_per_issue:,.0f}')
    print(f'  Total tokens (sample): {total_tokens_sample:,}')
    print(f'  Estimated total tokens (all issues): {total_tokens_all:,}')

print('\nDone!')


Estimating swiftlang/swift...
  Total closed issues: 10,248
  Sample size: 1000
  Avg tokens/issue (sample): 609
  Total tokens (sample): 609,287
  Estimated total tokens (all issues): 6,243,973

Done!


## Cost Analysis

In [13]:
# Create DataFrame
df = pd.DataFrame(results)
display(df)

# Calculate costs
print('\n' + '='*70)
print('EMBEDDING COSTS (text-embedding-3-small @ $0.02 per 1M tokens)')
print('='*70)

total_cost = 0

for _, row in df.iterrows():
    repo = row['repo']
    total_tokens = row['total_tokens_estimated']
    cost = (total_tokens / 1_000_000) * 0.02
    total_cost += cost
    
    print(f"\n{repo}:")
    print(f"  Total issues: {row['total_issues']:,}")
    print(f"  Total tokens (estimated): {total_tokens:,}")
    print(f"  Avg tokens/issue: {row['avg_tokens_per_issue']:,}")
    print(f"  **Cost: ${cost:,.2f}**")

print('\n' + '='*70)
print(f'TOTAL EMBEDDING COST: ${total_cost:,.2f}')
print('='*70)

,repo,total_issues,sample_size,avg_tokens_per_issue,total_tokens_sample,total_tokens_estimated
0,swiftlang/swift,10248,1000,609,609287,6243973



EMBEDDING COSTS (text-embedding-3-small @ $0.02 per 1M tokens)

swiftlang/swift:
  Total issues: 10,248
  Total tokens (estimated): 6,243,973
  Avg tokens/issue: 609
  **Cost: $0.12**

TOTAL EMBEDDING COST: $0.12


## Time Estimate

Rough timing estimates:

In [14]:
print("\nTime Estimates (rough):")
print('-' * 50)

for _, row in df.iterrows():
    repo = row['repo']
    total_issues = row['total_issues']
    
    # Estimate: ~1 second per issue for GitHub API + embedding
    # But batching and parallelization can improve this
    time_seconds = total_issues * 1  # 1 sec per issue
    time_minutes = time_seconds / 60
    time_hours = time_minutes / 60
    
    print(f"\n{repo}:")
    if time_hours >= 1:
        print(f"  {time_hours:.1f} hours (serial)")
    else:
        print(f"  {time_minutes:.1f} minutes (serial)")
    print(f"  With batching/parallelization: 10-25% of above")


Time Estimates (rough):
--------------------------------------------------

swiftlang/swift:
  2.8 hours (serial)
  With batching/parallelization: 10-25% of above


## Notes

- Token counts use tiktoken's `cl100k_base` encoding (used by text-embedding-3 models)
- Costs calculated using Batch API pricing ($0.02 per 1M tokens)
- Embedding timing estimate: ~1 second per issue (with overhead)
- With batch requests and parallelization: 10-25x faster
- GitHub API rate limits: 5,000 requests/hour (authenticated)